# E10：Pandas 筛选、清洗与变量生成

**课程**：Python 程序设计  
**适用对象**：应用统计学专业一年级  
**实验类型**：综合性实验  
**建议学时**：2 学时

## 实验目标

完成本实验后，你应当能够：

1. 使用 `loc`、`iloc` 和布尔条件选择记录。
2. 清理字符串字段并统一分类变量的取值。
3. 识别缺失值、转换数值类型并处理重复记录。
4. 使用 `np.where()` 和 `pd.cut()` 生成分析变量。
5. 核验并导出规范的学生调查主表。

## 实验信息

- 姓名：  
- 学号：  
- 班级：  
- 日期：

## 提交要求

1. 在本 Notebook 中完成全部任务，不删除题目、提示和课程提供的数据单元。
2. 保留最终可运行的代码、输出结果和必要的 Markdown 说明。
3. 不修改或覆盖 `lab/shared` 中的原始数据和官方检查点。
4. 提交前重启内核并按顺序运行全部单元，检查行数、字段、主键和缺失值口径。
5. 如使用帮助文档、代码补全或 AI 工具，应在实验小结中说明用途和核验方式。

## Jupyter 使用提示

- 先运行课程提供的路径和输入准备单元，再完成后续任务。
- 修改代码后必须重新运行当前单元，输出才会更新。
- DataFrame 结果异常时，先检查 `shape`、字段名、数据类型和缺失值。
- 如果变量状态混乱，重启内核并从上到下重新运行。

## 建议节奏

- 输入复核与筛选：15 分钟
- 字符串和类别清理：20 分钟
- 数值、缺失与重复处理：20 分钟
- 变量生成：15 分钟
- 核验和导出：10 分钟
- 调试、实验小结与提交：10 分钟

## 主线输入：把 E09 的审计结论转成清洗规则

E10 再次读取原始调查表，不依赖 E09 的内核状态。这样既保留章内连续性，也允许从原始数据独立重做。正式输出写入个人目录，并由固定规则核验。

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

chapter4_dir = Path(".").resolve()
notebook_names = ["e10.ipynb", "e10_inclass.ipynb"]
if not any((chapter4_dir / name).exists() for name in notebook_names):
    chapter4_dir = Path("lab/applied-stat-64/chapter4").resolve()

shared_dir = chapter4_dir.parents[1] / "shared"
raw_path = shared_dir / "raw" / "student_survey_raw.txt"
output_path = chapter4_dir / "e10_output" / "e10_students_clean.csv"

raw_df = pd.read_csv(raw_path)
print("输入文件：", raw_path)
print("原始行列数：", raw_df.shape)

## 任务 1：使用标签、位置和条件选择记录

完成三种选择：

1. 使用 `loc` 按标签选择前 3 行中的指定字段；
2. 使用 `iloc` 按位置选择前 3 行、前 4 列；
3. 临时把学习时长转换为数值，筛选学习时长不少于 8 小时的记录。

这里的临时转换只服务于筛选，正式清洗在后续任务完成。

In [ ]:
rows_by_label = ...
rows_by_position = ...
study_hours_probe = ...
high_hours_rows = ...

print(rows_by_label)
print(rows_by_position)
print("学习时长不少于 8 小时的原始记录数：", ...)

## 任务 2：清理字符串和分类变量

复制原始表为 `df`，去除所有文本字段的首尾空白并统一为小写类别。按照数据字典把类别别名映射为规范值，完成后检查每个分类变量的唯一取值。

In [ ]:
df = raw_df.copy()

text_columns = [...]
for column in text_columns:
    df[column] = ...

df["gender"] = ...
df["major"] = ...
df["prior_programming"] = ...
df["support_plan"] = ...

for column in ["gender", "major", "prior_programming", "support_plan"]:
    print(column, "：", ...)

## 任务 3：转换数值并保留缺失信息

把学习时长转换为数值。出勤率需要先去掉百分号，再转换为数值。无法转换的标记应成为缺失值，不进行均值填补或其他插补。

In [ ]:
df["reported_study_hours"] = ...
attendance_text = ...
df["reported_attendance_rate"] = ...

print(df.dtypes)
print("\n缺失值数量：")
print(...)

## 任务 4：处理重复记录

先报告清理后 `student_id` 的重复数量，再按 `student_id` 保留第一次出现的记录。排序并重置索引，核对处理前后的行数。

In [ ]:
rows_before = len(df)
duplicate_count = ...

df = ...

print("处理前行数：", rows_before)
print("重复记录数：", duplicate_count)
print("处理后行数：", ...)
print("student_id 唯一：", ...)

## 任务 5：生成课堂分析变量

在工作副本 `analysis_df` 中：

1. 使用 `pd.cut()` 按 E03–E04 的同一口径把学习时长分为 `low`、`medium` 和 `high`：小于 4 为 `low`，小于 8 为 `medium`，其余为 `high`；
2. 先筛选出勤率非缺失的记录，再使用 `np.where()` 把它们分为 `high` 和 `regular`；
3. 原始出勤率缺失的记录继续保留缺失，不归入正常类别。

这些变量用于练习，不写入本章的规范学生主表。

In [ ]:
analysis_df = df.copy()
analysis_df["study_level"] = pd.cut(
    analysis_df["reported_study_hours"],
    bins=...,
    labels=["low", "medium", "high"],
    right=False,
)

known_attendance = ...
analysis_df["attendance_level"] = pd.Series(
    pd.NA,
    index=analysis_df.index,
    dtype="string",
)
analysis_df.loc[
    known_attendance,
    "attendance_level",
] = np.where(
    analysis_df.loc[
        known_attendance,
        "reported_attendance_rate",
    ] >= 90,
    ...,
    ...,
)

print(analysis_df["study_level"].value_counts(dropna=False, sort=False))
print(analysis_df["attendance_level"].value_counts(dropna=False))

## 任务 6：核验并导出规范学生主表

核验记录数、主键、字段顺序、缺失数量和分类取值。所有检查通过后，把规范字段导出到个人输出目录。不要覆盖共享检查点。

In [ ]:
expected_columns = [
    "student_id",
    "alias",
    "gender",
    "major",
    "prior_programming",
    "support_plan",
    "reported_study_hours",
    "reported_attendance_rate",
]
clean_students = ...

checks = {
    "记录数为 160": ...,
    "student_id 唯一": ...,
    "字段顺序正确": ...,
    "学习时长缺失数为 8": ...,
    "出勤率缺失数为 4": ...,
    "gender 取值正确": ...,
    "major 取值正确": ...,
    "编程经历取值正确": ...,
    "支持方案取值正确": ...,
}

for check_name, passed in checks.items():
    print(check_name, "：", passed)
if not all(checks.values()):
    raise ValueError("清洗结果核验未通过，请返回前面的任务检查。")

output_path.parent.mkdir(parents=True, exist_ok=True)
...
print("已导出：", output_path)

## 实验小结

请用 Markdown 回答：

1. 为什么类别统一必须在分组统计之前完成？
2. 为什么缺失值不应自动归入 `regular` 或其他正常类别？
3. 为什么正式输出不包含本实验临时生成的所有分析变量？

### 实验小结

- ...
- ...
- ...